# Lab 05: Spark Streaming Fundamentals

## Objectives
- Understand Structured Streaming concepts and architecture
- Create streaming DataFrames from various sources (files, sockets, rate)
- Apply transformations to streaming data
- Perform windowed operations (tumbling, sliding, session)
- Implement watermarking for handling late data
- Manage output modes (append, complete, update)
- Configure triggers and checkpoints for fault tolerance
- Monitor streaming queries with Spark UI

## Domain Coverage
- **Structured Streaming — 10%**

## Estimates
- **Time:** ~90-120 minutes
- **Difficulty:** Intermediate
- **Prerequisites:** Lab 01 (Spark Fundamentals), Lab 02 (Data Loading & Transformation)

![Spark Streaming Diagram](../assets/diagrams/lab-05-spark-streaming.mmd)

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, count, avg, sum
from pyspark.sql.functions import current_timestamp, expr
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
import os
import time

In [ ]:
# Create Spark session for streaming
spark = SparkSession.builder \
    .appName("SparkStreamingFundamentals") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.streaming.checkpointLocation", "checkpoint/streaming") \
    .config("spark.sql.shuffle.partitions", "10") \
    .getOrCreate()

# Create checkpoint directory
os.makedirs("checkpoint/streaming", exist_ok=True)

# Verify Spark session
print(f"Spark version: {spark.version}")
print(f"Streaming checkpoint location: checkpoint/streaming")

## A. Structured Streaming Concepts

Structured Streaming treats streaming data as a continuously appended table. You write batch-like queries, and Spark runs them incrementally.

In [ ]:
# Key concepts explanation
print("""Structured Streaming Key Concepts:

1. Input Table: Unbounded stream of data treated as a table
2. Query: Standard batch query on the input table
3. Result Table: Continuously updated as new data arrives
4. Output: External sink where results are written
5. Output Mode: How results are written (append, complete, update)
6. Trigger: When to execute the query (processing time, once, continuous)
7. Checkpoint: Fault tolerance through state recovery
""")

## B. Creating Streaming Sources

Create streaming DataFrames from different sources.

In [ ]:
# Rate source (for testing)
# Generates data continuously with timestamp and value
rate_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .option("numPartitions", 2) \
    .load()

print("Rate source created (generates 10 rows per second)")
print(f"Schema: {rate_stream.schema}")

In [ ]:
# File source (monitor directory for new files)
# Create streaming directory
streaming_dir = "data/streaming/input"
os.makedirs(streaming_dir, exist_ok=True)

# Define schema for file data
schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("value", DoubleType(), True)
])

# Create file streaming source
file_stream = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .csv(streaming_dir)

print(f"File source created monitoring: {streaming_dir}")

## C. Basic Streaming Transformations

Apply transformations to streaming data.

In [ ]:
# Simple transformation on rate stream
transformed_stream = rate_stream \
    .withColumn("doubled", col("value") * 2) \
    .withColumn("is_even", (col("value") % 2) == 0)

print("Transformed stream created with doubled values and even/odd flag")

In [ ]:
# Filter streaming data
filtered_stream = rate_stream.filter(col("value") > 50)

print("Filtered stream created (values > 50)")

## D. Window Operations

Window operations aggregate data over time windows.

In [ ]:
# Tumbling window (non-overlapping)
windowed_stream = rate_stream \
    .groupBy(window(col("timestamp"), "5 seconds")) \
    .agg(count("*").alias("count"), avg("value").alias("avg_value"))

print("Tumbling window stream created (5-second windows)")

In [ ]:
# Sliding window (overlapping)
sliding_stream = rate_stream \
    .groupBy(window(col("timestamp"), "5 seconds", "2 seconds")) \
    .agg(count("*").alias("count"), sum("value").alias("total_value"))

print("Sliding window stream created (5-second windows, 2-second slide)")

## E. Watermarking

Watermarking handles late data and controls state size.

In [ ]:
# Add watermark to handle late data
watermarked_stream = rate_stream \
    .withWatermark("timestamp", "10 seconds") \
    .groupBy(window(col("timestamp"), "5 seconds")) \
    .agg(count("*").alias("count"))

print("Watermarked stream created (10-second watermark)")
print("Late data arriving within 10 seconds will be included")

## F. Output Modes

Different output modes determine how results are written to sinks.

In [ ]:
# Demonstrate output modes
print("""Output Modes:

1. Append Mode (default): Only new rows are added
   - Use with: Simple aggregations, filters
   - Not for: Stateful aggregations

2. Complete Mode: All rows are output
   - Use with: Global aggregations
   - Not for: Unbounded streams

3. Update Mode: Only changed rows are output
   - Use with: Stateful aggregations
   - Requires: Result table with unique keys
""")

## G. Writing Streaming Queries

Start streaming queries with different sinks and configurations.

In [ ]:
# Write to console (for debugging)
console_query = transformed_stream \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Console query started (will output every 5 seconds)")
print("Query ID:", console_query.id)
print("Query status:", console_query.status)

In [ ]:
# Write to memory (for testing and queries)
memory_query = windowed_stream \
    .writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("windowed_counts") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Memory query started")
print("Query ID:", memory_query.id)

In [ ]:
# Query the in-memory table
time.sleep(10)  # Wait for some data to be processed

print("Querying in-memory table:")
spark.sql("SELECT * FROM windowed_counts ORDER BY window DESC").show()

In [ ]:
# Write to file (for persistence)
output_dir = "data/streaming/output"
os.makedirs(output_dir, exist_ok=True)

file_query = filtered_stream \
    .writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", output_dir) \
    .option("checkpointLocation", "checkpoint/file_sink") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"File query started writing to: {output_dir}")
print("Query ID:", file_query.id)

## H. Query Management

Monitor and manage streaming queries.

In [ ]:
# List active queries
print("Active streaming queries:")
for query in spark.streams.active:
    print(f"  ID: {query.id}")
    print(f"  Name: {query.name}")
    print(f"  Status: {query.status}")
    print(f"  Progress: {query.progress}")
    print()

# Get query by ID
if spark.streams.active:
    query = spark.streams.active[0]
    print(f"Query details for ID {query.id}:")
    print(f"Last progress: {query.lastProgress}")
    print(f"Recent progress: {query.recentProgress}")

In [ ]:
# Stop all queries
print("Stopping all streaming queries...")
for query in spark.streams.active:
    print(f"Stopping query {query.id}...")
    query.stop()
    print(f"Query {query.id} stopped")

print("\nAll queries stopped")

## I. Advanced Streaming Features

Explore advanced streaming capabilities.

In [ ]:
# Session windows (dynamic windows based on activity gaps)
from pyspark.sql.functions import session_window

session_stream = rate_stream \
    .withWatermark("timestamp", "10 seconds") \
    .groupBy(session_window(col("timestamp"), "5 seconds")) \
    .agg(count("*").alias("count"))

print("Session window stream created (5-second inactivity gap)")

In [ ]:
# Multiple sinks (same query to multiple outputs)
# Note: This requires different output modes per sink
print("""Multiple Sinks Strategy:

1. Create multiple queries from the same source
2. Each query can have different transformations
3. Each query writes to a different sink
4. Useful for: Different aggregations, different output formats
""")

## Exam-Style Review

**Q1.** What is the difference between tumbling and sliding windows?
- A) They are identical
- B) Tumbling windows overlap, sliding windows don't
- C) Tumbling windows are non-overlapping, sliding windows overlap
- D) Sliding windows are only for streaming

**Q2.** What is the purpose of watermarking in Structured Streaming?
- A) To improve query performance
- B) To handle late data and control state size
- C) To compress data
- D) To encrypt sensitive data

**Q3.** Which output mode should you use for stateful aggregations?
- A) Append mode
- B) Complete mode
- C) Update mode
- D) Any mode

**Q4.** What is the role of checkpointing in streaming?
- A) To improve performance
- B) To provide fault tolerance and state recovery
- C) To compress output
- D) To monitor queries

### Answers
- **Q1: C** — Tumbling windows are non-overlapping fixed-size windows, while sliding windows overlap with a slide interval.
- **Q2: B** — Watermarking handles late data by specifying how long to wait for data and controls state size by dropping old state.
- **Q3: C** — Update mode is appropriate for stateful aggregations as it outputs only changed rows.
- **Q4: B** — Checkpointing provides fault tolerance by periodically writing state to durable storage for recovery after failures.

## Key Takeaways
- Structured Streaming treats streaming data as a continuously appended table
- Rate source is useful for testing streaming applications
- Window operations aggregate data over time windows (tumbling, sliding, session)
- Watermarking handles late data and manages state size
- Output modes determine how results are written (append, complete, update)
- Multiple sinks allow different aggregations and output formats
- Checkpointing provides fault tolerance and state recovery
- Monitor streaming queries using Spark UI and query management APIs

**Next:** [Lab 06 — Machine Learning with MLlib](chapter-06-machine-learning-mllib.ipynb)

In [ ]:
# Cleanup
spark.catalog.clearCache()
print("Lab 05 complete. Cache cleared.")
print("Note: Streaming directories and checkpoints remain for inspection.")